### # Superstore Sales: Data Preparation

**Goal:** inspect the raw Superstore sales data, fix the few issues that matter for the dashboard, and save a clean file for the Streamlit app.

**Why a separate notebook?** Cleaning is done once, outside the app, so the dashboard itself stays fast and simple.

## Load data

The file is not UTF-8 encoded, so it is read with `latin-1` encoding.

In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/Sample_Superstore.csv",encoding="latin-1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## First look

Checking the shape, column types, missing values and duplicate rows.

In [2]:
df.info()
df.duplicated().sum()


<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

np.int64(0)

## Findings

- 9,994 rows and 21 columns
- No missing values, no duplicate rows
- `Row ID` is unique for every row, so it will be the protected ID column in the editor
- `Order Date` and `Ship Date` are stored as text (month/day/year), so they must be converted to real dates for date filters and time charts
- `Postal Code` is stored as a number, so leading zeros are lost; it should be text
- `Country` has a single value (United States) and is not useful for filtering
- 1,871 rows have negative `Profit`; these are loss-making sales, not errors, so they stay in the data

## Cleaning

Only the changes needed for the dashboard:

1. Convert `Order Date` and `Ship Date` to datetime
2. Convert `Postal Code` to a 5-character text value
3. Save the result to `data/clean/superstore_clean.csv`

In [3]:
# 1. Convert dates (month/day/year)
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

# 2. Postal code as 5-character text
df["Postal Code"] = df["Postal Code"].astype(str).str.zfill(5)

# 3. Save the clean file
df.to_csv("../data/clean/superstore_clean.csv", index=False)

In [4]:
print(df[["Order Date", "Ship Date", "Postal Code"]].dtypes)
print(df["Order Date"].min())

Order Date     datetime64[us]
Ship Date      datetime64[us]
Postal Code               str
dtype: object
2014-01-03 00:00:00
